# Kidney Stone Classification — Hybrid CNN + Vision Transformer

This notebook contains the main end-to-end workflow for downloading the Kaggle dataset, preparing train/validation/test splits, training a CNN and a Vision Transformer (ViT), combining predictions with soft voting, and evaluating the ensemble.

## Before running
- Run in Google Colab with a suitable runtime (GPU recommended).
- Check that the Kaggle dataset folder structure matches the path expected in the dataset section.
- Review the dataset splitting logic to ensure there is no overlap or duplicate images across train, validation, and test sets.
- This cleaned notebook has not been executed as part of this cleanup. Reported metrics must be generated from actual test predictions.
- Synthetic ROC/precision–recall curve cells from the original notebook were excluded because they do not represent actual model predictions.

## Workflow
1. Install dependencies and set random seeds.
2. Download and inspect the dataset.
3. Prepare train, validation, and test folders.
4. Train the CNN and ViT models.
5. Evaluate both models and their soft-voting ensemble on the test set.


In [ ]:
# ============================================================
# KIDNEY STONE CLASSIFICATION - HYBRID CNN + ViT ENSEMBLE
# FIXED: Handles 'Train'/'Test' subfolder structure from Kaggle
# ============================================================

# ------------------- 1. Install & Import -------------------
!pip install -q fastai transformers datasets timm kagglehub

import os, shutil, numpy as np, torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from fastai.vision.all import *
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import kagglehub

# Set seed & device
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ------------------- 2. Download Dataset from Kaggle -------------------
print("\nDownloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("mdsyedahmmed/kidney-stone-dataset")
print(f"Dataset downloaded to: {dataset_path}")

images_root = os.path.join(dataset_path, "Dataset")
print(f"Image root folder: {images_root}")
print("Top-level contents:", os.listdir(images_root))

# ------------------- 3. Detect and Print Folder Structure -------------------
# This dataset has Train/ and Test/ with class subfolders inside
# e.g. Dataset/Train/Stone/, Dataset/Train/Normal/
# We'll merge all images per class, then re-split 70/15/15

def collect_images_by_class(root):
    """Walk root recursively, collect images grouped by class folder name."""
    class_images = {}
    for dirpath, dirnames, filenames in os.walk(root):
        folder_name = os.path.basename(dirpath)
        images = [os.path.join(dirpath, f) for f in filenames
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if images:
            # Use parent folder name as class if current folder is Train/Test
            if folder_name.lower() in ('train', 'test', 'val', 'dataset'):
                continue
            class_images.setdefault(folder_name, []).extend(images)
    return class_images

class_images = collect_images_by_class(images_root)
print("\nDetected classes and image counts:")
for cls, imgs in class_images.items():
    print(f"  {cls}: {len(imgs)} images")

if not class_images:
    raise RuntimeError(
        f"No class images found under {images_root}. "
        f"Please inspect the folder structure manually."
    )

# ------------------- 4. Split into Train/Val/Test -------------------
train_dir = "/content/data/train"
val_dir   = "/content/data/val"
test_dir  = "/content/data/test"
shutil.rmtree("/content/data", ignore_errors=True)

for cls, images in class_images.items():
    # 70% train, 15% val, 15% test
    train_imgs, temp = train_test_split(images, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp, test_size=0.5, random_state=42)

    for split_dir in [train_dir, val_dir, test_dir]:
        os.makedirs(os.path.join(split_dir, cls), exist_ok=True)

    for img in train_imgs:
        shutil.copy(img, os.path.join(train_dir, cls, os.path.basename(img)))
    for img in val_imgs:
        shutil.copy(img, os.path.join(val_dir,   cls, os.path.basename(img)))
    for img in test_imgs:
        shutil.copy(img, os.path.join(test_dir,  cls, os.path.basename(img)))

print(f"\nTrain images: {sum(len(files) for _,_,files in os.walk(train_dir))}")
print(f"Val images:   {sum(len(files) for _,_,files in os.walk(val_dir))}")
print(f"Test images:  {sum(len(files) for _,_,files in os.walk(test_dir))}")

# ------------------- 5. Data Transforms & Loaders -------------------
IMG_SIZE   = 224
BATCH_SIZE = 32
mean, std  = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_ds = ImageFolder(train_dir, transform=train_tf)
val_ds   = ImageFolder(val_dir,   transform=val_tf)
test_ds  = ImageFolder(test_dir,  transform=val_tf)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_ds.classes
num_classes  = len(class_names)
print("Classes:", class_names)

# ------------------- 6. Train XResNet-50 (CNN) -------------------
def get_xresnet50(num_classes):
    m = xresnet50(pretrained=True)
    m[-1] = nn.Linear(m[-1].in_features, num_classes)   # replace head safely
    return m

model_cnn = get_xresnet50(num_classes).to(device)
criterion  = nn.CrossEntropyLoss()
optim_cnn  = optim.Adam(model_cnn.parameters(), lr=1e-4)
NUM_EPOCHS = 10  # 10+ epochs recommended for 96%+ accuracy

print("\n--- Training XResNet-50 ---")
for epoch in range(NUM_EPOCHS):
    model_cnn.train()
    running_loss = 0.0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optim_cnn.zero_grad()
        loss = criterion(model_cnn(imgs), lbls)
        loss.backward()
        optim_cnn.step()
        running_loss += loss.item() * imgs.size(0)

    model_cnn.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, preds = torch.max(model_cnn(imgs), 1)
            total   += lbls.size(0)
            correct += (preds == lbls).sum().item()
    print(f"Epoch {epoch+1}: Loss={running_loss/len(train_ds):.4f}, Val Acc={100*correct/total:.2f}%")

# ------------------- 7. Train Vision Transformer (ViT) -------------------
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
model_vit  = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
).to(device)

class VitDataset(Dataset):
    def __init__(self, folder_ds, processor):
        self.ds        = folder_ds
        self.processor = processor
        # pre-build a simple inverse-normalize to get PIL back from tensor
        self.to_pil = transforms.ToPILImage()
        inv_std  = [1/s for s in std]
        inv_mean = [-m/s for m, s in zip(mean, std)]
        self.denorm = transforms.Compose([
            transforms.Normalize([0,0,0], inv_std),
            transforms.Normalize(inv_mean, [1,1,1]),
        ])
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        img_tensor, label = self.ds[idx]
        pil_img = self.to_pil(self.denorm(img_tensor).clamp(0, 1))
        enc = self.processor(images=pil_img, return_tensors="pt")
        return enc["pixel_values"].squeeze(0), label

train_vit_ds = VitDataset(train_ds, processor)
val_vit_ds   = VitDataset(val_ds,   processor)
test_vit_ds  = VitDataset(test_ds,  processor)

train_loader_vit = DataLoader(train_vit_ds, BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_vit   = DataLoader(val_vit_ds,   BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_vit  = DataLoader(test_vit_ds,  BATCH_SIZE, shuffle=False, num_workers=2)

optim_vit = optim.Adam(model_vit.parameters(), lr=2e-5)   # lower LR for ViT

print("\n--- Training Vision Transformer ---")
for epoch in range(NUM_EPOCHS):
    model_vit.train()
    running_loss = 0.0
    for pix_vals, lbls in tqdm(train_loader_vit, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        pix_vals, lbls = pix_vals.to(device), lbls.to(device)
        optim_vit.zero_grad()
        out  = model_vit(pixel_values=pix_vals, labels=lbls)
        loss = out.loss
        loss.backward()
        optim_vit.step()
        running_loss += loss.item() * pix_vals.size(0)

    model_vit.eval()
    correct = total = 0
    with torch.no_grad():
        for pix_vals, lbls in val_loader_vit:
            pix_vals, lbls = pix_vals.to(device), lbls.to(device)
            logits = model_vit(pix_vals).logits
            _, preds = torch.max(logits, 1)
            total   += lbls.size(0)
            correct += (preds == lbls).sum().item()
    print(f"Epoch {epoch+1}: Loss={running_loss/len(train_vit_ds):.4f}, Val Acc={100*correct/total:.2f}%")

# ------------------- 8. Soft-Voting Ensemble -------------------
def get_logits_labels(model, loader, is_vit=False):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Evaluating"):
            imgs = imgs.to(device)
            logits = model(imgs).logits if is_vit else model(imgs)
            all_logits.append(logits.cpu().numpy())
            all_labels.extend(lbls.numpy())
    return np.concatenate(all_logits), np.array(all_labels)

cnn_logits, true_labels = get_logits_labels(model_cnn, test_loader,     is_vit=False)
vit_logits, _           = get_logits_labels(model_vit, test_loader_vit, is_vit=True)

cnn_preds      = np.argmax(cnn_logits, axis=1)
vit_preds      = np.argmax(vit_logits, axis=1)
cnn_probs      = torch.softmax(torch.tensor(cnn_logits), dim=-1).numpy()
vit_probs      = torch.softmax(torch.tensor(vit_logits), dim=-1).numpy()
ensemble_preds = np.argmax(0.5 * cnn_probs + 0.5 * vit_probs, axis=1)

cnn_acc      = accuracy_score(true_labels, cnn_preds)
vit_acc      = accuracy_score(true_labels, vit_preds)
ensemble_acc = accuracy_score(true_labels, ensemble_preds)

print("\n" + "="*52)
print("FINAL TEST RESULTS")
print("="*52)
print(f"XResNet-50 (CNN)        : {cnn_acc*100:.2f}%")
print(f"Vision Transformer (ViT): {vit_acc*100:.2f}%")
print(f"Soft-Voting Ensemble    : {ensemble_acc*100:.2f}%")
print("="*52)

# ------------------- 9. Confusion Matrix -------------------
cm = confusion_matrix(true_labels, ensemble_preds)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.title("Ensemble Confusion Matrix")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45)
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()